# 第二讲：NumPy 数组运算、向量化操作与随机数生成## 为什么量化必须学 NumPy？假设你要计算 5000 只股票 x 10 年的日收益率。用 Python 的 `for` 循环逐行算——可能要跑几分钟甚至更久。用 NumPy 的向量化运算——**不到 1 秒**。差距来自底层实现：```Python 循环（慢）:                 NumPy 向量化（快）:for i in range(len(a)):           c = a + b    c[i] = a[i] + b[i]↑ Python 层循环 n 次               ↑ C 层循环 1 次  每次都有解释器开销                  无 Python 层面开销```### NumPy vs Python List —— 本质区别| 操作 | Python List | NumPy Array ||------|-------------|-------------|| `a + b` | 拼接两个列表 | 逐元素加法 || `a * 3` | 重复列表 3 次 | 每个元素 x 3 || `a ** 2` | ❌ 报错 | 每个元素求平方 |NumPy 把「整个数组」当作一个操作单元，而不是一个个元素。这就是**向量化**。**学习目标**- 掌握逐元素运算和广播机制- 理解 axis 参数的真相- 告别 Python 循环，拥抱向量化- 掌握布尔索引和条件筛选- 学会用 Generator API 生成随机数---

### 导入 NumPy`import numpy as np` —— 约定俗成的缩写，之后所有 `np.xxx` 都是在用 NumPy。`time` 库用来计时对比。

In [ ]:
import numpy as npimport timeimport sys

## 2.1 数组运算 —— NumPy 的核心魔法先看一个对比，建立直觉：同样的 `+` 和 `*`，Python 列表和 NumPy 数组的行为完全不同。

In [ ]:
a = [1, 2, 3]b = [4, 5, 6]print("Python list:")print("  a + b =", a + b)print("  a * 3 =", a * 3)a_np = np.array([1, 2, 3])b_np = np.array([4, 5, 6])print("\nNumPy array:")print("  a + b =", a_np + b_np)print("  a * 3 =", a_np * 3)print("  a ** 2 =", a_np ** 2)print("  np.sqrt(a) =", np.sqrt(a_np))

### 2.1.1 逐元素运算**所有算术运算符都是逐元素的。** 这是 NumPy 最基本的原则。比较运算也返回布尔数组（True/False），这在后面的条件筛选中会大量用到。标量（单个数字）运算时会自动「拉伸」到数组的每个元素——这是最简单的广播。

In [ ]:
a = np.array([1, 2, 3, 4, 5], dtype=np.float64)b = np.array([10, 20, 30, 40, 50], dtype=np.float64)print("=== 算术运算 ===")print("a:", a)print("b:", b)print()print("a + b  =", a + b)print("a - b  =", a - b)print("a * b  =", a * b)print("a / b  =", a / b)print("a ** 2 =", a ** 2)print()print("=== 比较运算（返回布尔数组）===")print("a > 2:", a > 2)print("a == b:", a == b)print()print("=== 与标量的运算（自动广播）===")print("a + 10:", a + 10)print("a * 2:", a * 2)print("a > 3:", a > 3)

### 2.1.2 广播机制 —— NumPy 最强大的特性当两个形状不同的数组运算时，NumPy 自动把小的「拉伸」成大的，不复制数据，效率极高。#### 三条广播规则1. 从尾部维度开始比较（右对齐）2. 两个维度**相等**，或其中一个为 **1**，则兼容3. 不满足 → 报错 `ValueError`#### 图解```Case 1: (3, 4) 与标量 10  3x4 矩阵 + 一个数 10 → 10 被自动复制成 3x4 矩阵  这和我们刚看到的 a + 10 是一回事Case 2: (3, 4) 与 (3, 1)  (3, 1) 的列被复制 4 份 → 变成 (3, 4)Case 3: (3, 4) 与 (4,)  一维数组 (4,) 先变成 (1, 4)，再复制 3 行 → (3, 4)```> ⚠️ **常见坑：** `(3, 4)` 与 `(3, 3)` 不兼容——最后一维 4 ≠ 3，且都不是 1。

In [ ]:
data = np.array([[1, 2, 3, 4],                 [5, 6, 7, 8],                 [9, 10, 11, 12]], dtype=np.float64)print("原始数据 (3x4):")print(data)row_means = data.mean(axis=1, keepdims=True)print(f"\n行均值 shape: {row_means.shape}")print(row_means)centered = data - row_meansprint("\n中心化后（每行均值为 0）:")print(centered)print(f"\n验证每行均值: {centered.mean(axis=1)}")print("\n---\n")weights = np.array([0.1, 0.2, 0.3, 0.4])weighted = data * weightsprint("加权后（每列乘以不同的系数）:")print(weighted)print("\n---\n")x = np.array([1, 2, 3])y = np.array([10, 20, 30, 40])outer = x[:, np.newaxis] * y[np.newaxis, :]print("外积 (3x4):")print(outer)try:    a = np.ones((3, 4))    b = np.ones((3, 3))    c = a + bexcept ValueError as e:    print(f"\n广播失败示例: {e}")

### 2.1.3 聚合运算 —— axis 参数的真相**口诀：`axis` 指定的是「被消灭」的维度。**- `axis=0`：沿行方向压缩 → 结果中「行」消失了 → 每列求一个值- `axis=1`：沿列方向压缩 → 结果中「列」消失了 → 每行求一个值`keepdims=True` 保留被压缩的维度（变成 1），方便后续继续广播。#### 两种写法完全等价| 方法形式 | 函数形式 | |----------|----------|| `a.sum()` | `np.sum(a)` || `a.mean()` | `np.mean(a)` || `a.std()` | `np.std(a)` |

In [ ]:
arr = np.array([[1, 2, 3],                [4, 5, 6],                [7, 8, 9]])print("原始数组 (3x3):")print(arr)print()print("axis=0 (沿行压缩 → 每列一个值):")print("  sum:", arr.sum(axis=0))print("  mean:", arr.mean(axis=0))print()print("axis=1 (沿列压缩 → 每行一个值):")print("  sum:", arr.sum(axis=1))print("  mean:", arr.mean(axis=1))print()print("=== keepdims=True 的作用 ===")without = arr.sum(axis=1)with_kd = arr.sum(axis=1, keepdims=True)print(f"  sum(axis=1):            shape={without.shape}")print(f"  sum(axis=1, keepdims):   shape={with_kd.shape}")print()normalized = arr / arr.sum(axis=1, keepdims=True)print("每行除以该行和（归一化）:")print(normalized)

In [ ]:
arr = np.array([[1, 3, 5], [2, 4, 6]], dtype=np.float64)print("数组:", arr, f"shape={arr.shape}")print()print(f"sum:    {arr.sum()}")print(f"mean:   {arr.mean():.2f}")print(f"std:    {arr.std():.2f}")print(f"min:    {arr.min()}")print(f"max:    {arr.max()}")print(f"argmax: {arr.argmax()}")print(f"cumsum: {arr.cumsum()}")

### 2.1.4 线性代数运算（选学）> ⚠️ 初学 NumPy 可以跳过这一节。线性代数在资产组合优化中会用到，不是入门必备。**关键区分：**- `a * b` → 逐元素乘法（element-wise）- `a @ b` → 矩阵乘法（matrix multiplication）`@` 运算符是 Python 3.5+ 引入的，等价于 `np.dot(a, b)` 和 `np.matmul(a, b)`。

In [ ]:
A = np.array([[1, 2, 3],              [4, 5, 6]])B = np.array([[7, 8],              [9, 10],              [11, 12]])print("矩阵乘法 A @ B (2x3 @ 3x2 = 2x2):")print(A @ B)print()u = np.array([1, 2, 3])v = np.array([4, 5, 6])print(f"向量点积 u@v = {u @ v} = {np.dot(u, v)}")

In [ ]:
from numpy import linalg as LAM = np.array([[2, 1],              [1, 2]], dtype=np.float64)print("矩阵 M:")print(M)print()M_inv = LA.inv(M)print("M 的逆矩阵:")print(M_inv)print(f"验证 M @ M_inv = I:\n{M @ M_inv}")print()eigenvalues, eigenvectors = LA.eig(M)print(f"特征值: {eigenvalues}")print(f"行列式 det(M) = {LA.det(M):.1f}")

## 2.2 向量化操作 —— 告别 Python 循环### 什么是向量化？用**数组级别的操作**代替显式的 Python 循环。当你在 NumPy 中写 `a + b` 时，循环发生在编译好的 C 代码层，Python 解释器的每次循环开销全部被省掉了。### 到底快多少？实测

In [ ]:
n = 1_000_000a_list = list(range(n))b_list = list(range(n))t0 = time.perf_counter()c_loop = [a_list[i] + b_list[i] for i in range(n)]t_loop = time.perf_counter() - t0a_np = np.arange(n)b_np = np.arange(n)t0 = time.perf_counter()c_vec = a_np + b_npt_vec = time.perf_counter() - t0print(f"Python 列表推导式: {t_loop*1000:.1f} ms")print(f"NumPy 向量化:      {t_vec*1000:.1f} ms")print(f"加速: {t_loop/t_vec:.0f}x")print()print("向量化的代码更短、更可读、更快——没有理由不用它。")

### 2.2.2 通用函数（ufunc）ufunc 是 NumPy 中用 C 实现的逐元素函数，速度极快。每个算术运算符背后都有一个 ufunc（`+` → `np.add`，`*` → `np.multiply`）。| 类型 | 例子 ||------|------|| 一元 ufunc | `np.sqrt`, `np.exp`, `np.log`, `np.sin` || 二元 ufunc | `np.add`, `np.multiply`, `np.maximum`, `np.power` |

In [ ]:
x = np.array([0, 1, 2, 3, 10])print("=== 数学函数 ===")print("np.sqrt(x):", np.sqrt(x))print("np.exp(x):", np.exp(x))print("np.log(x+1):", np.log(x + 1))print("\n=== 三角函数 ===")angles = np.array([0, np.pi/2, np.pi])print("np.sin:", np.sin(angles))print("np.cos:", np.cos(angles))print("\n=== 取整/符号 ===")vals = np.array([-2.7, -1.2, 0.0, 1.2, 2.7])print("np.floor:", np.floor(vals))print("np.ceil:", np.ceil(vals))print("np.round:", np.round(vals))print("np.sign:", np.sign(vals))

In [ ]:
a = np.array([1, 2, 3, 4])b = np.array([10, 20, 30, 40])print("np.add(a,b) =", np.add(a, b))print("np.multiply(a,b) =", np.multiply(a, b))print("np.maximum(a,b) =", np.maximum(a, b))print("np.minimum(a,b) =", np.minimum(a, b))c = np.empty(4)np.add(a, b, out=c)print("np.add(a,b,out=c):", c)

### 2.2.3 布尔索引与条件筛选这是 NumPy 中**最常用的数据筛选方式**。量化里无处不在：- 选出 PE < 15 的股票- 筛选成交量 > 1亿 的交易日- 找出价格突破均线的日期#### 关键规则```pythondata[data > 30]        # 筛选大于 30 的元素data[(data > 10) & (data < 50)]   # 用 & 不是 and！data[(data < 5) | (data > 90)]    # 用 | 不是 or！data[~(data > 30)]     # 用 ~ 不是 not！```> ⚠️ **常见坑：** 组合条件时用 `&` `|` `~`，不是 `and` `or` `not`。而且每个条件外面要加括号！

In [ ]:
data = np.array([15, 8, 42, 23, 7, 91, 56, 4, 33, 18])print("原始数据:", data)mask = data > 30print("\ndata > 30 的掩码:", mask)print("筛选结果:", data[mask])print("\n=== 组合条件 ===")print("10 < x < 50:", data[(data > 10) & (data < 50)])print("x < 10 或 x > 80:", data[(data < 10) | (data > 80)])print("不大于30:", data[~(data > 30)])

In [ ]:
prices = np.array([100, 200, 150, 300, 250])print("np.where 根据条件替换值:")labels = np.where(prices > 180, "买入", "观望")print(labels)print("\nnp.where 返回满足条件的索引:")buy_indices = np.where(prices > 180)[0]print(f"价格 > 180 的位置: {buy_indices}")print(f"对应的价格: {prices[buy_indices]}")

## 2.3 随机数生成### 先理解：计算机的「随机数」其实是确定的给定相同的**种子（seed）**，产生完全相同的随机序列。这不是 bug——科学计算需要**可复现**。你告诉别人种子，他们就能完全复现你的分析。### 新旧写法对比```旧: np.random.seed(42);  np.random.rand(3)    # 全局状态，多线程可能互相干扰新: rng = np.random.default_rng(42);  rng.random(3)   # 每个 rng 独立```类比：旧写法是全家人共用一个遥控器（互相干扰），新写法是每人一个遥控器。

In [ ]:
rng = np.random.default_rng(seed=42)print("rng.random(5):", rng.random(5))rng2 = np.random.default_rng(seed=42)print("rng2.random(5):", rng2.random(5))print("完全相同?", np.array_equal(rng.random(5), rng2.random(5)))print("\n不同种子 = 不同序列:")for s in [42, 123, 2024]:    rng = np.random.default_rng(seed=s)    print(f"  seed={s:>4}: {rng.random(5)}")

### 三大常用分布| 方法 | 分布 | 参数 | 量化用途 ||------|------|------|----------|| `rng.uniform(low, high)` | 均匀分布 | 下界（含）、上界（不含） | 等概率抽样 || `rng.normal(loc, scale)` | 正态分布 | loc=均值, scale=标准差 | 收益率建模 || `rng.integers(low, high)` | 随机整数 | 下界（含）、上界（默认不含） | 掷骰子、选股编号 |> ⚠️ `integers` 默认 `endpoint=False`，所以 `integers(1, 7)` 是 1~6。想包含 7 要写 `endpoint=True`。

In [ ]:
rng = np.random.default_rng(seed=42)print("uniform [0,1):", rng.uniform(0, 1, 5))print("uniform [-5,5):", rng.uniform(-5, 5, 5))print("\nnormal N(0,1):", rng.normal(0, 1, 5))print("normal N(5,2):", rng.normal(5, 2, 5))print("\nintegers [1,7):", rng.integers(1, 7, 10))print("integers [1,6] 含两端:", rng.integers(1, 6, 10, endpoint=True))

## 2.4 小结 & 自检清单| 技能 | ✓ ||------|---|| 理解 Python List vs NumPy Array 的本质区别 | ☐ || 逐元素运算：所有运算符都是逐元素的 | ☐ || 广播机制：三条规则，`(3,4)+(3,1)` 可行，`(3,4)+(3,3)` 报错 | ☐ || `axis=0` 压缩行（每列一个值），`axis=1` 压缩列（每行一个值） | ☐ || 向量化：C 层循环，比 Python 循环快数十到数百倍 | ☐ || 布尔索引：`&` `|` `~`，不是 `and` `or` `not` | ☐ || `np.where(条件, A, B)` = 向量化的 if-else | ☐ || 随机数：用 `default_rng(seed)` 保证可复现 | ☐ |### 📝 你的笔记区新建 Markdown Cell，写下回答：- 用你自己的话解释「广播机制」——像一个 10 岁小孩解释- `axis=0` 和 `axis=1` 的区别，用一句话说清楚- NumPy 比 Python 循环快的原因---**下一讲：** 随机数与蒙特卡洛模拟。